# 损失函数

In [5]:
import numpy as np

## 均方误差

均方误差（Mean Squared Error, MSE）：

$$E = \frac{1}{2} \sum_{k} (y_k - t_k)^2$$

- $y_k$：神经网络的输出（预测值）
- $t_k$：监督数据（真实值）
- 系数 $\frac{1}{2}$ 是为了求导后系数变为 1，方便计算

In [6]:
import numpy as np

In [7]:
def mean_squared_error(y, t):
    return 0.5 * np.sum((y - t) ** 2)

## 交叉熵误差

交叉熵误差（Cross Entropy Error）：

$$E = -\sum_{k} t_k \log y_k$$

- $y_k$：神经网络的输出（softmax 后的概率）
- $t_k$：监督数据的 one-hot 标签（正确类为 1，其余为 0）
- 因 $t_k$ 是 one-hot，实际只计算正确类那一项，即 $E = -\log(\text{正确类的预测概率})$
- 代码中加 $\delta = 10^{-7}$ 防止 $\log 0 = -\infty$

In [8]:
def cross_entropy_error(y, t):
    delta = 1e-7
    return -np.sum(t * np.log(y + delta))

## mini-batch 版交叉熵误差

训练时通常不会一次只计算 1 条数据，而是取一小批数据（mini-batch）一起计算损失。

假设一个 batch 有 $N$ 条数据，则交叉熵误差取平均值：

$$E = -\frac{1}{N}\sum_{n}\sum_{k} t_{nk}\log y_{nk}$$

$t_{nk}$表示第$n$个数据的第$k$个元素的值，$y_{nk}$是神经网络的输出，$t_{nk}$是监督数据

- $y$ 的形状是 `(batch_size, class_num)`，表示每条数据属于各类别的预测概率
- $t$ 可以是 one-hot 标签，形状同样是 `(batch_size, class_num)`
- $t$ 也可以是普通标签数组，例如 `[2, 7, 0]`，表示每条数据的正确类别下标
- 最后除以 `batch_size`，得到这个 mini-batch 的平均损失

In [9]:
def cross_entropy_error_batch(y, t):
    """mini-batch 版交叉熵误差。

    y: 神经网络输出的概率，形状可以是 (class_num,) 或 (batch_size, class_num)
    t: 正确标签，可以是 one-hot，也可以是类别下标
    """
    delta = 1e-7

    # 单条数据时，统一转换成 batch_size=1 的二维数组
    if y.ndim == 1:
        y = y.reshape(1, y.size)
        t = t.reshape(1, t.size) if t.ndim == 1 else t.reshape(1)

    batch_size = y.shape[0]

    # one-hot 标签转换为正确类别下标
    if t.ndim == 2:
        t = np.argmax(t, axis=1)

    return -np.sum(np.log(y[np.arange(batch_size), t] + delta)) / batch_size

下面两种标签写法等价：一种使用 one-hot，另一种直接使用正确类别下标。

In [10]:
y = np.array([
    [0.1, 0.8, 0.1],
    [0.7, 0.2, 0.1],
    [0.2, 0.3, 0.5],
])

t_one_hot = np.array([
    [0, 1, 0],
    [1, 0, 0],
    [0, 0, 1],
])
t_label = np.array([1, 0, 2])

print(cross_entropy_error_batch(y, t_one_hot))
print(cross_entropy_error_batch(y, t_label))

0.42432173598526096
0.42432173598526096
